# Calibração— MiDaS 

## O que este notebook faz
1. Lê depth sintético MiDaS e depth real TUM do Drive
2. Calcula calibração  global: `d_real = s × d_sint + t`
3. Gera depth corrigido e associations prontos para ORB-SLAM3

#


In [ ]:
#  Montar Drive e instalar dependências
from google.colab import drive
drive.mount('/content/drive')

!pip install scipy scikit-learn -q

import os, json, shutil
import numpy as np
from PIL import Image
from scipy.stats import theilslopes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

print('Dependências OK!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dependências OK!


In [ ]:
# CONFIGURAÇÕES

DATASET = 'fr3_office'          # 'fr2_xyz' | 'fr1_desk'
MODELO  = 'midas'            # 'midas' 
METODO  = 'affine'           # 'affine' | 'escala' | 'robusto'

# Fator de conversão do depth real TUM
DEPTH_FACTOR_REAL = {
    'fr2_xyz':   5208.0,   # freiburg2 calibrado
    'fr1_desk':  5000.0,   # freiburg1 padrão
    'fr3_office': 5000.0,  # freiburg3 padrão
}

# Fator usado ao salvar o depth sintético
DEPTH_FACTOR_SINT = {
    'midas':            5000.0,   # normalizado [0,1] × 5000
}

DATASET_CONFIG = {
    'fr2_xyz': {
        'folder':      'rgbd_dataset_freiburg2_xyz',
        'drive_midas': '/content/drive/MyDrive/orbslam_midas/fr2_xyz',
        'drive_dav2':  '/content/drive/MyDrive/orbslam_dav2_metric/fr2_xyz',
    },
    'fr1_desk': {
        'folder':      'rgbd_dataset_freiburg1_desk',
        'drive_midas': '/content/drive/MyDrive/orbslam_midas/fr1_desk',
        'drive_dav2':  '/content/drive/MyDrive/orbslam_dav2_metric/fr1_desk',
    },
    'fr3_office': {
        'folder':      'rgbd_dataset_freiburg3_long_office_household',
        'drive_midas': '/content/drive/MyDrive/orbslam_midas/fr3_office',
        'drive_dav2':  '/content/drive/MyDrive/orbslam_dav2_metric/fr3_office',
    },
}

cfg          = DATASET_CONFIG[DATASET]
DATASET_DIR  = f'/content/{cfg["folder"]}'
DRIVE_SRC    = cfg['drive_midas'] if MODELO == 'midas' else cfg['drive_dav2']
DRIVE_OUT    = f'{DRIVE_SRC}/affine'
os.makedirs(DRIVE_OUT, exist_ok=True)

DF_REAL = DEPTH_FACTOR_REAL[DATASET]
DF_SINT = DEPTH_FACTOR_SINT[MODELO]

SINT_FOLDER  = f'depth_{MODELO}'
AFFINE_FOLDER = f'depth_{MODELO}_affine'

print(f'Dataset      : {DATASET}')
print(f'Modelo       : {MODELO}')
print(f'Método       : {METODO}')
print(f'DF real      : {DF_REAL}')
print(f'DF sint      : {DF_SINT}')
print(f'Drive src    : {DRIVE_SRC}')
print(f'Drive out    : {DRIVE_OUT}')

Dataset      : fr3_office
Modelo       : midas
Método       : affine
DF real      : 5000.0
DF sint      : 5000.0
Drive src    : /content/drive/MyDrive/orbslam_midas/fr3_office
Drive out    : /content/drive/MyDrive/orbslam_midas/fr3_office/affine


In [ ]:
# Extrair dataset e depth sintético do Drive

# Extrair dataset TUM (depth real + rgb)
tgz_urls = {
    'fr2_xyz':  'https://cvg.cit.tum.de/rgbd/dataset/freiburg2/rgbd_dataset_freiburg2_xyz.tgz',
    'fr1_desk': 'https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz',
    'fr3_office': 'https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_long_office_household.tgz',
}

if not os.path.exists(DATASET_DIR):
    print(f'Baixando {DATASET}...')
    !wget -q --show-progress {tgz_urls[DATASET]} -O /content/dataset.tgz
    !tar -xzf /content/dataset.tgz -C /content/
    !rm /content/dataset.tgz
    print('Dataset extraído!')
else:
    print(f'Dataset já existe: {DATASET_DIR}')

# Extrair depth sintético do Drive
SINT_DIR = os.path.join(DATASET_DIR, SINT_FOLDER)
if not os.path.exists(SINT_DIR):
    zip_name = f'depth_{MODELO}_{DATASET}.zip'
    zip_path = os.path.join(DRIVE_SRC, 'processed', zip_name)
    if not os.path.exists(zip_path):
        zip_path = os.path.join(DRIVE_SRC, zip_name)
    print(f'Extraindo {zip_name}...')
    !unzip -q {zip_path} -d {DATASET_DIR}/
    print('Depth sintético extraído!')
else:
    print(f'Depth sintético já existe: {SINT_DIR}')

REAL_DIR = os.path.join(DATASET_DIR, 'depth')
print(f'\nDepth real  : {len(os.listdir(REAL_DIR))} arquivos')
print(f'Depth sint  : {len(os.listdir(SINT_DIR))} arquivos')

Baixando fr3_office...
/content/dataset.tg 100%[===================>]   1.38G  24.1MB/s    in 47s     
Dataset extraído!
Extraindo depth_midas_fr3_office.zip...
Depth sintético extraído!

Depth real  : 2509 arquivos
Depth sint  : 2585 arquivos


In [ ]:
# Inverter disparidade MiDaS nos uint16

import os
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# Pasta com depth MiDaS original
SINT_DIR_ORIG = os.path.join(DATASET_DIR, 'depth_midas')

# Pasta de saída invertida
SINT_DIR_INV  = os.path.join(DATASET_DIR, 'depth_midas_inv')
os.makedirs(SINT_DIR_INV, exist_ok=True)

print(f'Invertendo disparidade MiDaS...')
arquivos = sorted(f for f in os.listdir(SINT_DIR_ORIG) if f.endswith('.png'))

for fname in tqdm(arquivos, desc='Invertendo'):
    img = np.array(Image.open(os.path.join(SINT_DIR_ORIG, fname))).astype(np.float32)

    # Inverter: depth = max - disparidade (inversão simples)
    img_inv = img.max() - img

    # Normalizar para usar o range completo
    d_min, d_max = img_inv.min(), img_inv.max()
    if d_max - d_min > 1e-6:
        img_inv = (img_inv - d_min) / (d_max - d_min) * 5000

    Image.fromarray(img_inv.astype(np.uint16)).save(
        os.path.join(SINT_DIR_INV, fname))

print(f'Frames invertidos: {len(arquivos)}')
print(f'Pasta: {SINT_DIR_INV}')

# Atualizar SINT_DIR para usar a versão invertida nas próximas células
SINT_DIR = SINT_DIR_INV
print(f'SINT_DIR atualizado para: {SINT_DIR}')

Invertendo disparidade MiDaS...


Invertendo:   0%|          | 0/2585 [00:00<?, ?it/s]

Frames invertidos: 2585
Pasta: /content/rgbd_dataset_freiburg3_long_office_household/depth_midas_inv
SINT_DIR atualizado para: /content/rgbd_dataset_freiburg3_long_office_household/depth_midas_inv


In [ ]:
# Parear frames por timestamp e amostrar pixels

MIN_M, MAX_M, TS_TOL_MS = 0.3, 8.0, 50.0

def listar_ts(pasta):
    out = {}
    for f in os.listdir(pasta):
        if f.endswith('.png'):
            try: out[float(f[:-4])] = f
            except: pass
    return out

sint_ts  = listar_ts(SINT_DIR)
real_ts  = listar_ts(REAL_DIR)
real_keys = np.array(sorted(real_ts.keys()))

# Parear por timestamp mais próximo
pares = []
for ts_s in sorted(sint_ts.keys())[::3]:
    i = int(np.argmin(np.abs(real_keys - ts_s)))
    if abs(ts_s - real_keys[i]) * 1000 <= TS_TOL_MS:
        pares.append((sint_ts[ts_s], real_ts[real_keys[i]]))
print(f'Frames pareados: {len(pares)}')

# Amostrar pixels válidos
d_s_all, d_r_all = [], []
for f_s, f_r in tqdm(pares[:400], desc='Amostrando pixels'):
    d_s = np.array(Image.open(os.path.join(SINT_DIR, f_s))).astype(np.float32) / DF_SINT
    d_r = np.array(Image.open(os.path.join(REAL_DIR, f_r))).astype(np.float32) / DF_REAL
    if d_s.shape != d_r.shape:
        d_s = np.array(Image.fromarray(d_s).resize(
            (d_r.shape[1], d_r.shape[0]), Image.BILINEAR))
    mask = (d_r > MIN_M) & (d_r < MAX_M) & (d_s > 0)
    if mask.sum() < 100: continue
    idx = np.where(mask.ravel())[0]
    if len(idx) > 400: idx = np.random.choice(idx, 400, replace=False)
    d_s_all.append(d_s.ravel()[idx])
    d_r_all.append(d_r.ravel()[idx])

d_s = np.concatenate(d_s_all)
d_r = np.concatenate(d_r_all)
print(f'Pixels amostrados: {len(d_s)}')

Frames pareados: 862


Amostrando pixels:   0%|          | 0/400 [00:00<?, ?it/s]

Pixels amostrados: 160000


In [ ]:
# Calcular calibração 


def fit_escala(s_, r_):
    s = float(np.median(r_ / np.clip(s_, 1e-6, None)))
    return s, 0.0

def fit_affine(s_, r_):
    A = np.vstack([s_, np.ones_like(s_)]).T
    sol, *_ = np.linalg.lstsq(A, r_, rcond=None)
    return float(sol[0]), float(sol[1])

def fit_robusto(s_, r_):
    idx = np.random.choice(len(s_), min(20000, len(s_)), replace=False)
    res = theilslopes(r_[idx], s_[idx])
    return float(res[0]), float(res[1])

def metricas(pred, gt):
    pred  = np.clip(pred, 1e-3, None)
    absrel = float(np.mean(np.abs(pred - gt) / gt))
    rmse   = float(np.sqrt(np.mean((pred - gt)**2)))
    delta1 = float(np.mean(np.maximum(pred/gt, gt/pred) < 1.25))
    return absrel, rmse, delta1

ajustes = {
    'escala':  fit_escala(d_s, d_r),
    'affine':  fit_affine(d_s, d_r),
    'robusto': fit_robusto(d_s, d_r),
}

print(f'{'método':<12}{'AbsRel':>9}{'RMSE(m)':>10}{'δ1':>8}   (s, t)')
print('-' * 58)
a, r, d1 = metricas(d_s, d_r)
print(f'{'original':<12}{a:>9.3f}{r:>10.3f}{d1:>8.3f}')
for nome, (s, t) in ajustes.items():
    a, r, d1 = metricas(s * d_s + t, d_r)
    print(f'{nome:<12}{a:>9.3f}{r:>10.3f}{d1:>8.3f}   (s={s:.4f}, t={t:.4f})')

# Parâmetros do método escolhido
S, T = ajustes[METODO]
print(f'\n→ Usando {METODO}: s={S:.4f}, t={T:.4f}')

método         AbsRel   RMSE(m)      δ1   (s, t)
----------------------------------------------------------
original        0.695     1.802   0.018
escala          0.521     1.043   0.365   (s=3.9851, t=0.0000)
affine          0.452     1.015   0.379   (s=3.0680, t=0.3733)
robusto         0.385     1.093   0.285   (s=2.1338, t=0.5036)

→ Usando affine: s=3.0680, t=0.3733


In [ ]:
#  Aplicar calibração e salvar depth corrigido

AFFINE_DIR = os.path.join(DATASET_DIR, AFFINE_FOLDER)
os.makedirs(AFFINE_DIR, exist_ok=True)

associations = []
arquivos     = sorted(f for f in os.listdir(SINT_DIR) if f.endswith('.png'))

print(f'Aplicando affine (s={S:.4f}, t={T:.4f})...')
for fname in tqdm(arquivos, desc='Gerando depth corrigido'):
    img = np.array(Image.open(os.path.join(SINT_DIR, fname))).astype(np.float32)

    # Converter para metros
    d_m = img / DF_SINT

    # Aplicar affine
    d_corr = np.clip(S * d_m + T, 0, 10.0)

    # Converter de volta para uint16 (× 5000)
    out = (d_corr * 5000).astype(np.uint16)
    Image.fromarray(out).save(os.path.join(AFFINE_DIR, fname))

    ts = fname[:-4]
    associations.append(
        f'{ts} rgb/{fname} {ts} {AFFINE_FOLDER}/{fname}'
    )

print(f'Frames gerados: {len(associations)}')

# Inspecionar
print('\n=== DEPTH CORRIGIDO (primeiros 3) ===')
for fname in arquivos[:3]:
    img = np.array(Image.open(os.path.join(AFFINE_DIR, fname)))
    print(f'  {fname}: mean={img.mean()/5000:.2f}m  max={img.max()/5000:.2f}m')

print('\n=== DEPTH REAL TUM (primeiros 3) ===')
for fname in sorted(os.listdir(REAL_DIR))[:3]:
    img = np.array(Image.open(os.path.join(REAL_DIR, fname)))
    print(f'  {fname}: mean={img.mean()/DF_REAL:.2f}m  max={img.max()/DF_REAL:.2f}m')

Aplicando affine (s=3.0680, t=0.3733)...


Gerando depth corrigido:   0%|          | 0/2585 [00:00<?, ?it/s]

Frames gerados: 2585

=== DEPTH CORRIGIDO (primeiros 3) ===
  1341847980.722988.png: mean=2.00m  max=3.44m
  1341847980.754743.png: mean=1.99m  max=3.44m
  1341847980.786856.png: mean=1.98m  max=3.44m

=== DEPTH REAL TUM (primeiros 3) ===
  1341847980.723020.png: mean=2.00m  max=9.33m
  1341847980.754755.png: mean=2.08m  max=9.87m
  1341847980.786879.png: mean=2.07m  max=9.87m


In [ ]:
# Salvar no Drive

# Associations
assoc_name  = f'associations_{MODELO}_affine.txt'
assoc_local = os.path.join(DATASET_DIR, assoc_name)
with open(assoc_local, 'w') as f:
    f.write('\n'.join(associations))
shutil.copy(assoc_local, os.path.join(DRIVE_OUT, assoc_name))
print(f'Associations: {assoc_name} ({len(associations)} pares)')

# Stats JSON
a, r, d1 = metricas(S * d_s + T, d_r)
stats = {
    'dataset':  DATASET,
    'modelo':   MODELO,
    'metodo':   METODO,
    's':        S,
    't':        T,
    'absrel':   a,
    'rmse_m':   r,
    'delta1':   d1,
    'frames':   len(associations),
    'df_sint_entrada':  DF_SINT,
    'df_saida': 5000.0,
    'nota':     f'd_corr = {S:.4f} * d_sint + {T:.4f}  (em metros)'
}
stats_path = os.path.join(DRIVE_OUT, f'stats_{DATASET}_{MODELO}_affine.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Stats: {json.dumps(stats, indent=2)}')

# ZIP depth corrigido
print(f'\nCompactando {AFFINE_FOLDER}...')
zip_path = os.path.join(DRIVE_OUT, f'depth_{MODELO}_affine_{DATASET}')
shutil.make_archive(zip_path, 'zip', DATASET_DIR, AFFINE_FOLDER)
print(f'ZIP: {zip_path}.zip')

print('\n=== DRIVE ===')
!ls -lh {DRIVE_OUT}

Associations: associations_midas_affine.txt (2585 pares)
Stats: {
  "dataset": "fr3_office",
  "modelo": "midas",
  "metodo": "affine",
  "s": 3.067995071411133,
  "t": 0.3732604384422302,
  "absrel": 0.45206937193870544,
  "rmse_m": 1.015328049659729,
  "delta1": 0.37866875,
  "frames": 2585,
  "df_sint_entrada": 5000.0,
  "df_saida": 5000.0,
  "nota": "d_corr = 3.0680 * d_sint + 0.3733  (em metros)"
}

Compactando depth_midas_affine...
ZIP: /content/drive/MyDrive/orbslam_midas/fr3_office/affine/depth_midas_affine_fr3_office.zip

=== DRIVE ===
total 451M
-rw------- 1 root root 261K Jun 27 15:02 associations_midas_affine.txt
-rw------- 1 root root 150K Jun 27 14:58 calibracao_fr3_office_midas.pdf
-rw------- 1 root root 450M Jun 27 15:03 depth_midas_affine_fr3_office.zip
-rw------- 1 root root  342 Jun 27 15:02 stats_fr3_office_midas_affine.json
